In [1]:
import sys
from pyprojroot import here

sys.path.insert(0, str(here()))

import pandas as pd
import numpy as np
import xarray as xr
import pickle
import pymc as pm
import pytensor.tensor as pt
import matplotlib.pyplot as plt
import arviz as az
import pymc_extras as pmx
import geopandas as gpd

import pandas_datareader.data as pdr
import datetime

In [5]:
#Load predictions
df = pd.read_csv(here('data/climate_forecast.csv'))
# Transform to USD millions
df['real_gdp'] =  df['real_gdp'] *1e-6
df =df[['time', 'real_gdp']]

# Transform time to datetime format
df['time'] = pd.to_datetime(df['time'])
df = df.set_index('time')


# load CATDSGE damages
losses_df = pd.read_csv(here('data/cat_dsge_gdp_losses.csv'))
losses_df['Year'] = pd.to_datetime(losses_df['Year'])

losses_df = losses_df.set_index('Year')
# losses_df = losses_df / 100

In [6]:
losses_df

,50 RY,40 RY,30 RY,25 RY,20 RY,10 RY,4 RY
Year,,,,,,,
2015-01-01,-3.69,-2.59,-1.87,-1.52,-1.18,-0.41,-0.07
2025-01-01,-8.58,-6.67,-4.93,-3.92,-2.95,-1.22,-0.23
2035-01-01,-15.32,-11.31,-8.17,-6.66,-5.25,-1.96,-0.42
2045-01-01,-18.48,-14.86,-10.74,-8.77,-6.78,-2.78,-0.57
2055-01-01,-20.49,-16.29,-11.92,-10.06,-8.25,-3.53,-0.77


In [3]:
# Select GDP for interest years
dates = losses_df.index.to_list()

df = df.loc[dates]

In [9]:
# Create value losses
value_losses = pd.DataFrame(  index = dates, columns = losses_df.columns.to_list() )

for col in losses_df.columns.to_list():
    value_losses[col] = losses_df[col] / df['real_gdp'] * 100

In [10]:
value_losses

,50 RY,40 RY,30 RY,25 RY,20 RY,10 RY,4 RY
2015-01-01,-0.025557,-0.017938,-0.012952,-0.010528,-0.008173,-0.002840,-0.000485
2025-01-01,-0.039429,-0.030652,-0.022656,-0.018014,-0.013557,-0.005607,-0.001057
2035-01-01,-0.054556,-0.040276,-0.029094,-0.023717,-0.018696,-0.006980,-0.001496
2045-01-01,-0.053764,-0.043232,-0.031246,-0.025515,-0.019725,-0.008088,-0.001658
2055-01-01,-0.050814,-0.040398,-0.029561,-0.024948,-0.020459,-0.008754,-0.001910


In [11]:
value_losses.to_csv(here('data/cat_dsge_gdp_loses_percentage.csv'))